In [ ]:
import sys

import igris_c_sdk as igc_sdk

In [ ]:
domain_id = 0
namespace = "igris_c_IG01"  # set to your robot's namespace (see README "Connecting to the robot")

channel_instance = igc_sdk.ChannelFactory.Instance()
channel_instance.Init(domain_id, namespace)

client = igc_sdk.IgrisC_Client()
client.Init()
client.SetTimeout(20.0)

lowstate_subscriber = igc_sdk.LowStateSubscriber("lowstate")
lowstate_cnt = 0

In [ ]:
def lowstate_callback(msg):
    imu_state = msg.imu_state()
    orientation = imu_state.orientation()  # geometry_msgs Quaternion: x/y/z/w accessors
    angular_velocity = imu_state.angular_velocity()  # geometry_msgs Vector3: x/y/z
    linear_acceleration = imu_state.linear_acceleration()  # geometry_msgs Vector3: x/y/z

    motor_states = msg.motor_state()
    motor_1 = motor_states[0]
    motor_31 = motor_states[30]

    q = motor_1.q()
    dq = motor_1.dq()
    tau_est = motor_1.tau_est()
    temp = motor_1.temperature()
    status = motor_1.status_bits()

    joint_states = msg.joint_state()
    joint_1 = joint_states[0]
    joint_31 = joint_states[30]
    q = joint_1.q()
    dq = joint_1.dq()
    tau_est = joint_1.tau_est()
    status_bits = joint_1.status_bits()

In [ ]:
lowstate_subscriber.init(lowstate_callback)

In [ ]:
del lowstate_subscriber

In [ ]:
lowcmd_publisher = igc_sdk.LowCmdPublisher("lowcmd")
lowcmd_publisher.init()

In [ ]:
def call_init_pdu(client: igc_sdk.IgrisC_Client, pdu_init_type: int, type_name: str):
    print(f"Calling Init with {type_name}...")
    res = client.InitPdu(pdu_init_type, 5000)

    result = "SUCCESS" if res.success() else "FAILED" + " - " + res.message()
    print(f"Init {type_name} result: {result}")


def call_set_torque(client: igc_sdk.IgrisC_Client, torque_type: int, type_name: str):
    print(f"Calling Set Torque with {type_name}...")
    res = client.SetTorque(torque_type, 5000)

    result = "SUCCESS" if res.success() else "FAILED" + " - " + res.message()
    print(f"Set Torque {type_name} result: {result}")


def call_send_control_mode_command(
    client: igc_sdk.IgrisC_Client,
    cmd_type: int,
    mode_name: str,
    preset_id: str = "",
    is_cyclic: bool = False,
):
    print(f"Calling SendControlModeCommand with {mode_name}...")
    res = client.SendControlModeCommand(cmd_type, preset_id, is_cyclic, 5000)

    result = "SUCCESS" if res.success() else "FAILED" + " - " + res.message()
    print(f"SendControlModeCommand {mode_name} result: {result}")

In [ ]:
call_init_pdu(client, igc_sdk.PDU_INIT, "PDU_INIT")

In [ ]:
call_init_pdu(client, igc_sdk.PDU_OFF, "PDU_OFF")

In [ ]:
call_init_pdu(client, igc_sdk.MOTOR_INIT, "MOTOR_INIT")

In [ ]:
call_init_pdu(client, igc_sdk.PDU_AND_MOTOR_INIT, "PDU_AND_MOTOR_INIT")

In [ ]:
call_set_torque(client, igc_sdk.TORQUE_OFF, "TORQUE_OFF")

In [ ]:
call_set_torque(client, igc_sdk.TORQUE_ON, "TORQUE_ON")

In [ ]:
call_send_control_mode_command(
    client,
    igc_sdk.CONTROL_MODE_CMD_LOW_LEVEL_JOINT_CONTROL,
    "CONTROL_MODE_CMD_LOW_LEVEL_JOINT_CONTROL",
)

In [ ]:
call_send_control_mode_command(
    client,
    igc_sdk.CONTROL_MODE_CMD_HIGH_LEVEL_JOINT_CONTROL,
    "CONTROL_MODE_CMD_HIGH_LEVEL_JOINT_CONTROL",
)

In [ ]:
low_cmd_msg = igc_sdk.LowCmd_()
for i in range(0, 31):
    if i == 30:
        motor_cmd = low_cmd_msg.motors()[i]
        motor_cmd.q(0.3)
        motor_cmd.dq(0.0)
        motor_cmd.tau(0.0)
        motor_cmd.kp(5)
        motor_cmd.kd(0.1)
        motor_cmd.id(i)
    elif i == 29:
        motor_cmd = low_cmd_msg.motors()[i]
        motor_cmd.q(0.0)
        motor_cmd.dq(0.0)
        motor_cmd.tau(0.0)
        motor_cmd.kp(2)
        motor_cmd.kd(0.05)
        motor_cmd.id(i)
    else:
        motor_cmd = low_cmd_msg.motors()[i]
        motor_cmd.q(0.0)
        motor_cmd.dq(0.0)
        motor_cmd.tau(0.0)
        motor_cmd.kp(0.0)
        motor_cmd.kd(0.0)
        motor_cmd.id(i)
# kinematic_modes is a per-group array of 5 (waist / L_ankle / R_ankle / L_wrist / R_wrist)
low_cmd_msg.kinematic_modes([igc_sdk.KINEMATIC_MODE_MS] * 5)
lowcmd_publisher.write(low_cmd_msg)